In [ ]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer

load_dotenv()


In [ ]:
llm = ChatOllama(
    model="qwen2.5-coder:7b",
    base_url="http://127.0.0.1:11434",
    temperature=0.9,
)
response = llm.invoke("What is the capital of France?")
print(response.content)


In [ ]:
!uv pip install requests beautifulsoup4

To extract individual debate

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

def get_debate_urls_from_volume(volume_url):
    """
    Parses the volume landing index layout (Image 2 & 3)
    and collects links to every individual debate date.
    """
    print(f"[*] Analyzing index page layout at: {volume_url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(volume_url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Error fetching volume index page: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    debate_links = []
    
    # Try locating links within the list layout elements
    # We look for anchor tags that match typical date slugs
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        # Debates URLs on this platform generally follow a /debates/DD-mmm-YYYY pattern
        if '/debates/' in href:
            full_url = urljoin(volume_url, href)
            if full_url not in debate_links:
                debate_links.append(full_url)
                
    print(f"[+] Found {len(debate_links)} debate link sittings under this volume.")
    return sorted(list(set(debate_links)))

def extract_transcript_from_page(debate_url):
    """
    Parses the multi-column component view (Image 1 & 4) 
    and returns parsed, structured conversations and titles.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(debate_url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Skipping {debate_url}: Error fetching page ({e})")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    page_data = []
    
    # Extract structural details (Volume title and sitting date)
    title_text = "Unknown Debate Heading"
    title_elem = soup.find('h1') or soup.find('h2')
    if title_elem:
        title_text = title_elem.text.strip()
        
    page_data.append(f"==================================================")
    page_data.append(f"DEBATE URL: {debate_url}")
    page_data.append(f"HEADING: {title_text}")
    page_data.append(f"==================================================\n")

    # Target content area holding block coordinates
    # Looking for conversational rows or standardized block classes matching the layout tables
    paragraphs = soup.find_all(['p', 'div'], class_=lambda c: c and ('speaker' in c or 'debate' in c or 'text' in c))
    
    # Fallback if custom layout classes aren't explicitly caught by conditional string filters:
    if not paragraphs:
        content_area = soup.find('article') or soup.find('main') or soup.body
        paragraphs = content_area.find_all(['p', 'h3', 'h4'])

    current_speaker = "CHAIRMAN / HOUSE NOTE"
    
    for p in paragraphs:
        text = p.text.strip()
        if not text:
            continue
            
        # Ignore social links or site footer metadata elements
        if any(term in text for term in ["Copy Link", "Email", "Facebook", "Twitter", "LinkedIn"]):
            continue

        # Look for explicit speaker indicators (Bolds / Column headers)
        strong_tag = p.find(['strong', 'b'])
        if strong_tag:
            pot_speaker = strong_tag.text.strip().rstrip(':')
            if len(pot_speaker) < 60: # Threshold ensures it's a structural name string, not text
                current_speaker = pot_speaker
                speech_text = text[len(strong_tag.text):].strip().lstrip(':').strip()
            else:
                speech_text = text
        else:
            # If line starts with a parenthetical note like (The Secretary then called out...)
            if text.startswith('(') and text.endswith(')'):
                current_speaker = "HOUSE NOTE"
                speech_text = text
            else:
                speech_text = text

        if speech_text:
            # Enforce formatting requirement requested: "speaker : text"
            page_data.append(f"{current_speaker} : {speech_text}")
            
    page_data.append("\n\n") # Distinct spacing buffer between historical dates
    return "\n".join(page_data)

def main():
    # Insert any Volume URL matching the Layout Index from Image 2 / 3
    volume_index_url = "https://www.constitutionofindia.net/constituent-assembly-debate/volume-1/"
    
    # Master txt collection file output target
    output_txt_file = "volume_1_debates_transcript.txt"
    
    # Phase 1: Dynamic Discovery
    debate_urls = get_debate_urls_from_volume(volume_index_url)
    
    if not debate_urls:
        print("[!] No active paths captured. Terminating scraper run.")
        return

    # Phase 2: Iterative Extraction loop
    with open(output_txt_file, "w", encoding="utf-8") as outfile:
        outfile.write(f"--- CONSTITUENT ASSEMBLY TRANSLATION LOG ---\n")
        outfile.write(f"Source Volume Directory Index: {volume_index_url}\n\n")
        
        for idx, url in enumerate(debate_urls, 1):
            print(f"[*] Processing sitting ({idx}/{len(debate_urls)}): {url}")
            
            extracted_text = extract_transcript_from_page(url)
            if extracted_text:
                outfile.write(extracted_text)
                print(f"[+] Successfully appended content data.")
            
            # Defensive execution spacing to keep server overhead light
            time.sleep(1.5)
            
    print(f"\n[Done] All extracted data output compiled into single file target: '{output_txt_file}'")

if __name__ == "__main__":
    main()

to extract all the debates and volumes

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re

def get_all_volume_urls(main_url):
    """
    Parses the main archive landing page to fetch all Volume URLs.
    """
    print(f"[*] Extracting volume directories from: {main_url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(main_url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Master Page Connection Error: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    volumes = []
    
    # Target anchor tags containing volume link pathways
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href'].lower()
        if 'volume-' in href or '/volume/' in href:
            full_url = urljoin(main_url, a_tag['href'])
            if full_url not in [v['url'] for v in volumes]:
                vol_match = re.search(r'volume[-/](\d+)', href)
                vol_num = vol_match.group(1) if vol_match else len(volumes) + 1
                volumes.append({
                    "number": vol_num,
                    "url": full_url
                })
                
    # Sort sequentially to run cleanly from Volume 1 onwards
    volumes.sort(key=lambda x: int(x['number']))
    return volumes

def get_debate_urls_from_volume(volume_url):
    """
    Parses the volume landing index layout (Image 2 & 3)
    and collects links to every individual debate date.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(volume_url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Error fetching volume index page: {e}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    debate_links = []
    
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        if '/debates/' in href:
            full_url = urljoin(volume_url, href)
            if full_url not in debate_links:
                debate_links.append(full_url)
                
    return sorted(list(set(debate_links)))

def extract_transcript_from_page(debate_url):
    """
    Parses the multi-column component view (Your exact original function layout 
    with alignment safety patches applied to block menu bleeding).
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(debate_url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Skipping {debate_url}: Error fetching page ({e})")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    page_data = []
    
    title_text = "Unknown Debate Heading"
    title_elem = soup.find('h1') or soup.find('h2')
    if title_elem:
        title_text = title_elem.text.strip()
        
    page_data.append(f"==================================================")
    page_data.append(f"DEBATE URL: {debate_url}")
    page_data.append(f"HEADING: {title_text}")
    page_data.append(f"==================================================\n")

    paragraphs = soup.find_all(['p', 'div'], class_=lambda c: c and ('speaker' in c or 'debate' in c or 'text' in c))
    
    if not paragraphs:
        content_area = soup.find('article') or soup.find('main') or soup.body
        paragraphs = content_area.find_all(['p', 'h3', 'h4'])

    current_speaker = "CHAIRMAN / HOUSE NOTE"
    
    for p in paragraphs:
        text = p.text.strip()
        if not text:
            continue
            
        # --- FIXED ALIGNMENT FILTER BLOCKS ---
        # 1. Skip breadcrumb links or structural separators completely
        if "≫" in text or "≫" in p.text or any(term in text for term in ["Debates", "Volume"]) and len(text) < 40:
            continue
            
        # 2. Ignore meta actions/social wrappers
        if any(term in text for term in ["Copy Link", "Email", "Facebook", "Twitter", "LinkedIn", "Table of contents", "VIEW ALL VOLUMES"]):
            continue

        strong_tag = p.find(['strong', 'b'])
        if strong_tag:
            pot_speaker = strong_tag.text.strip().rstrip(':')
            
            # 3. Check if the bold text is actually a metadata item header rather than a person talking
            if len(pot_speaker) < 60 and pot_speaker != "PROGRAMME OF BUSINESS": 
                current_speaker = pot_speaker
                speech_text = text[len(strong_tag.text):].strip().lstrip(':').strip()
            else:
                # If it's an agenda heading title, display it cleanly instead of masking it as a speaker name
                if pot_speaker == "PROGRAMME OF BUSINESS":
                    page_data.append(f"\n[SECTION: {pot_speaker}]\n")
                speech_text = text
        else:
            if text.startswith('(') and text.endswith(')'):
                current_speaker = "HOUSE NOTE"
                speech_text = text
            else:
                speech_text = text

        if speech_text:
            # Prevent metadata remnants from leaking into your speaker rows
            if speech_text == title_text or speech_text == "CONSTITUENT ASSEMBLY DEBATES":
                continue
            page_data.append(f"{current_speaker} : {speech_text}")
            
    page_data.append("\n\n") 
    return "\n".join(page_data)

def scrape_entire_archive():
    """
    Iterative wrapper control logic that loops sequentially 
    through all individual volumes and dumps separate output logs.
    """
    master_landing_url = "https://www.constitutionofindia.net/constitution-assembly-debates/"
    
    # Find all volume tracks sequentially
    volumes = get_all_volume_urls(master_landing_url)
    
    if not volumes:
        print("[!] No active volume tracks captured. Terminating program.")
        return

    for vol in volumes:
        vol_num = vol["number"]
        vol_url = vol["url"]
        output_txt_file = f"Volume_{vol_num}.txt"
        
        print(f"\n--- Processing Thread Initiated: VOLUME {vol_num} ---")
        print(f"[*] Extracting debate pages for volume index: {vol_url}")
        
        debate_urls = get_debate_urls_from_volume(vol_url)
        print(f"[+] Found {len(debate_urls)} session entries inside Volume {vol_num}")
        
        if not debate_urls:
            continue
            
        # Stream content logs chronologically directly to their respective file
        with open(output_txt_file, "w", encoding="utf-8") as outfile:
            outfile.write(f"==================================================\n")
            outfile.write(f"             CONSTITUENT ASSEMBLY DEBATES        \n")
            outfile.write(f"                       VOLUME {vol_num}          \n")
            outfile.write(f"==================================================\n\n")
            
            for idx, url in enumerate(debate_urls, 1):
                print(f"    [{idx}/{len(debate_urls)}] Extracting layout stream: {url}")
                extracted_text = extract_transcript_from_page(url)
                if extracted_text:
                    outfile.write(extracted_text)
                
                # Execution safety break
                time.sleep(1.0)
                
        print(f"[Done] Complete dataset written to text file: '{output_txt_file}'")
        time.sleep(1.5)

if __name__ == "__main__":
    scrape_entire_archive()

to scrape constitutions 

In [ ]:
import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# =========================================================
# CONFIG
# =========================================================
BASE_URL = "https://www.constitutionofindia.net/historical-constitutions/"
OUTPUT_DIR = "constitutions_txt"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    )
}

# =========================================================
# CREATE OUTPUT DIRECTORY
# =========================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# GET ALL CONSTITUTION LINKS
# =========================================================

def get_constitution_links():
    """
    Scrapes all constitution page links from the historical constitutions page.
    """

    response = requests.get(BASE_URL, headers=HEADERS)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    constitution_links = set()

    # Find all anchor tags
    for a in soup.find_all("a", href=True):

        href = a["href"]

        # Convert relative URLs to absolute
        full_url = urljoin(BASE_URL, href)

        # Filter valid constitution pages
        if (
            "/historical-constitution/" in full_url
            and full_url != BASE_URL
        ):
            constitution_links.add(full_url)

    return sorted(list(constitution_links))


# =========================================================
# CLEAN TEXT
# =========================================================

def clean_text(text):
    """
    Cleans excessive whitespace and blank lines.
    """

    lines = text.splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if line:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


# =========================================================
# SCRAPE SINGLE CONSTITUTION PAGE
# =========================================================

def scrape_constitution(url):
    """
    Scrapes the complete constitution text from a page.
    """

    print(f"Scraping: {url}")

    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # -----------------------------------------------------
    # TITLE
    # -----------------------------------------------------

    title_tag = soup.find("h1")

    if title_tag:
        title = title_tag.get_text(strip=True)
    else:
        title = "Untitled"

    # -----------------------------------------------------
    # MAIN CONTENT
    # -----------------------------------------------------

    # This site stores article content inside page builders,
    # so we target the main content area.

    content_div = soup.find("main")

    if not content_div:
        content_div = soup.body

    # Remove unwanted elements
    for tag in content_div.find_all([
        "script",
        "style",
        "nav",
        "header",
        "footer",
        "button",
        "svg",
        "img",
        "aside"
    ]):
        tag.decompose()

    text = content_div.get_text(separator="\n")

    cleaned = clean_text(text)

    return title, cleaned


# =========================================================
# SAVE TO TXT
# =========================================================

def save_to_txt(title, text):

    # Safe filename
    filename = "".join(
        c for c in title if c.isalnum() or c in (" ", "-", "_")
    ).rstrip()

    filepath = os.path.join(
        OUTPUT_DIR,
        f"{filename}.txt"
    )

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"Saved: {filepath}")


# =========================================================
# MAIN
# =========================================================

def main():

    links = get_constitution_links()

    print(f"\nFound {len(links)} constitution pages\n")

    for link in links:

        try:
            title, text = scrape_constitution(link)

            save_to_txt(title, text)

            # polite delay
            time.sleep(1)

        except Exception as e:
            print(f"Error scraping {link}")
            print(e)
            print("-" * 50)


if __name__ == "__main__":
    main()


Found 34 constitution pages

Scraping: https://www.constitutionofindia.net/historical-constitution/a-scheme-of-political-safeguards-for-the-protection-of-the-depressed-classes-in-the-future-constitution-of-a-self-governing-india-a-memorandum-by-dr-ambedkar-and-rao-bahadur-r-srinivasan-1930/
Saved: constitutions_txt/A Scheme of Political Safeguards for the Protection of the Depressed Classes in the Future Constitution of a Self-Governing India - A Memorandum by Dr Ambedkar and Rao Bahadur R Srinivasan 1930.txt
Scraping: https://www.constitutionofindia.net/historical-constitution/aundh-state-constitution-act-1939/
Saved: constitutions_txt/Aundh State Constitution Act 1939.txt
Scraping: https://www.constitutionofindia.net/historical-constitution/cabinet-mission-plan-cabinet-mission-1946/
Saved: constitutions_txt/Cabinet Mission Plan Cabinet Mission 1946.txt
Scraping: https://www.constitutionofindia.net/historical-constitution/communal-deadlock-and-a-way-to-solve-it/
Saved: constitutions_

In [2]:
!uv pip install chromadb

Using Python 3.14.5 environment at: /Users/apple/dev/.venv
Checked 1 package in 38ms
